In [1]:
import sys
sys.path.append("..")

import os
import numpy as np
import wfdb
from src.pipeline import AAMI_MAP

DATA_DIR = "../data/mitdb"
OUT_DIR = "../app/samples"
os.makedirs(OUT_DIR, exist_ok=True)

SECONDS = 60

def save_sample(rec):
    record = wfdb.rdrecord(f"{DATA_DIR}/{rec}")
    ann = wfdb.rdann(f"{DATA_DIR}/{rec}", "atr")
    fs = record.fs
    sig = record.p_signal[:, record.sig_name.index("MLII")]

    # beats of class S or V (the interesting ones)
    rare = np.array([s for s, sym in zip(ann.sample, ann.symbol)
                     if AAMI_MAP.get(sym) in ("S", "V")])

    win = SECONDS * fs
    starts = np.arange(0, len(sig) - win, 10 * fs)
    counts = [((rare >= s) & (rare < s + win)).sum() for s in starts]
    best = starts[int(np.argmax(counts))]

    np.savetxt(f"{OUT_DIR}/record_{rec}.csv", sig[best:best + win],
               header="MLII_mV", comments="", fmt="%.4f")
    print(rec, "| start:", best // fs, "s | S/V beats in window:", max(counts))

for rec in ["100", "200", "233"]:
    save_sample(rec)

100 | start: 830 s | S/V beats in window: 5
200 | start: 1090 s | S/V beats in window: 46
233 | start: 110 s | S/V beats in window: 40
